# R-Traffic: GPS and 5G timing advance traces

The `R-Traffic` dataset contains combined GPS and 5G base station (gNB) traces
collected in the vicinity of the Ericsson buildings in Lund, Sweden. For each
record the file gives

* the position of the user equipment (UE) from GPS,
* the speed of the UE,
* radio measurements from the gNB, most importantly the *timing advance* (TA), and
* a label for the movement scenario: walking, biking, driving, or stuck in a
  traffic jam.

Timing advance is the offset with which the UE must advance its uplink
transmission so that it arrives at the gNB in time. The offset is set by the
propagation time of the radio signal between the UE and the gNB, and hence by
the distance between them, under line-of-sight conditions. The landscape around
the Ericsson buildings is flat, so this assumption holds for this dataset.

The intended use of the dataset is to train a model that determines how a UE is
moving in the landscape, based on the radio measurements. The ground truth
position and the movement labels come from the GPS traces and from notes taken
during the collection.

In this tutorial we will

* load the data from the data portal,
* visualize the trajectories on an OpenStreetMap road network,
* look at the speed and the movement labels in the time domain,
* explore the timing advance measurements, and
* build simple classifiers for the movement labels.

In [ ]:
%pip install -q git+https://github.com/wara-ops/DataportalClient.git@${DATAPORTAL_CLIENT_VERSION}

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

from dataportal import DataportalClient

plt.rcParams['figure.figsize'] = [10, 6]

## Loading the data

To access the data we use the data portal client together with a personal
token: paste your token inside the quotes in the cell below (see the
*PortalBasics* tutorial if you need to obtain one). If you are in the Jupyter
environment provided by the portal and the environment variable
`DATAPORTAL_CLIENT_TOKEN` is set, you can leave `token` empty instead.

In [ ]:
# paste your token inside the quotes (or leave empty if DATAPORTAL_CLIENT_TOKEN is set)
token = ''
dataset = 'R-Traffic'
client = DataportalClient(token)

Each file in the dataset is a set of traces from one measurement day. The list
below shows the files in the dataset, with the time span and the number of
records for each.

In [ ]:
files = client.fromDataset(dataset).listFiles()
for f in files:
    print(f)

We load one of the files into a pandas DataFrame. Pick a `FileID` from the list
above; we use the first one here (2025-12-08, about 27 minutes of traces).

In [ ]:
file_id = 100107
df = client.getData(file_id)
print(df.shape)
df.head()

## The data

The DataFrame has one row per radio record. The columns can be grouped into

* **time**: `timestamp` (Unix time, s), `datetime` (local time)
* **position**: `lat`/`lon` (raw GPS), `true_lat`/`true_lon` (ground truth), and
  their Cartesian versions in meters, `glue_x`/`glue_y` and `x`/`y`, in the
  Swedish SWEREF99 coordinate system
* **speed**: `speed` and `speed_smooth` (km/h, derived from the position)
* **map**: `seg_id` (road segment id from OpenStreetMap), `carway`/`bikeway`/
  `walkway` (flags for the type of road segment), `speed_limit` (km/h)
* **radio**: `radionode010` (distance to the gNB, m), `distanceTa` (distance
  estimated from the TA, m), the timing advance measurements (`taMeas`,
  `taMeasHighRes`, `filteredTA`, `scaledTA`, `timingOffsetFloat`,
  `taObservability`, `accumulatedTae`, ...), and `trace` (which radio record
  the row comes from)
* **label**: `label` (0 walk, 1 bike, 2 car, 3 jam) and the one-hot columns
  `walk`, `bike`, `car`, `jam`

The labels come from notes taken during the data collection:

* `walk(0)` - walking
* `bike(1)` - driving a car at low speed, 10-15 km/h, on roads where bikes are
  allowed to drive
* `car(2)` - normal driving
* `jam(3)` - very low speed, or stopped, in a traffic jam

Note that `bike(1)` is *not* a bicycle, but a car driven slowly on a bike road.

In [ ]:
df.info(memory_usage='deep')

In [ ]:
print('Records per label:')
df.label.value_counts().sort_index()

## Where was the trace collected?

The traces were collected in the Brunnshög area of Lund, around the Ericsson
buildings, where the gNB (named `radionode 010` in the data) is located. The
file `osm/roads.geojson` in this directory contains the road network of the
area, extracted from [OpenStreetMap](https://www.openstreetmap.org/)
(copyright the OpenStreetMap contributors, licensed under ODbL). We use it to
draw the trajectory of the UE.

In [ ]:
# The road network sits next to this notebook, in the osm/ subdirectory.
# (If you cloned the tutorial repo and run the notebook from outside it,
# it is in DCT-R-Traffic/osm/.)
osm_file = next(
    (p for p in (Path('osm/roads.geojson'), Path('DCT-R-Traffic/osm/roads.geojson')) if p.exists()),
    None,
)
if osm_file is None:
    raise FileNotFoundError(
        'osm/roads.geojson not found. Run the notebook from the directory '
        'containing R-Traffic_example.ipynb or from its parent directory.'
    )
roads = json.loads(osm_file.read_text())
print(f'{len(roads["features"])} road segments from OpenStreetMap')

In [ ]:
# Local projection: equirectangular in meters, centered on the middle of the area.
def to_xy(lat, lon, lat0=55.7220, lon0=13.2295):
    x = (lon - lon0) * 111320 * np.cos(np.radians(lat0))
    y = (lat - lat0) * 110574
    return x, y

fig, ax = plt.subplots(figsize=(11, 11))

# Zoom in on the trajectory, with some margin. The limits are also used
# to keep the street name labels inside the map.
x_all, y_all = to_xy(df.true_lat, df.true_lon)
margin = 150
xlim = (x_all.min() - margin, x_all.max() + margin)
ylim = (y_all.min() - margin, y_all.max() + margin)

# Draw the road network, thicker and darker for larger roads. A street can
# consist of many segments, so each name is labeled at most once, and only
# if the label position is inside the map.
style = {
    'motorway':      (4.0, '0.70'), 'motorway_link': (3.0, '0.70'),
    'primary':       (3.0, '0.75'), 'secondary':     (2.5, '0.80'),
    'tertiary':      (1.8, '0.60'), 'residential':   (1.2, '0.55'),
    'unclassified':  (1.0, '0.55'), 'living_street': (1.0, '0.55'),
    'pedestrian':    (1.0, '0.45'), 'service':       (0.8, '0.70'),
    'cycleway':      (0.9, '0.30'),
    'path':          (0.5, '0.85'), 'footway':       (0.5, '0.85'),
    'steps':         (0.5, '0.85'), 'track':         (0.5, '0.85'),
    'bridleway':     (0.5, '0.85'),
}
labeled = set()
for feat in roads['features']:
    highway = feat['properties'].get('highway', '')
    lw, color = style.get(highway, (0.8, '0.75'))
    coords = np.array(feat['geometry']['coordinates'])
    x, y = to_xy(coords[:, 1], coords[:, 0])
    ax.plot(x, y, color=color, lw=lw, zorder=1)
    name = feat['properties'].get('name')
    if name and lw >= 1.8 and len(coords) >= 3 and name not in labeled:
        mid = len(coords) // 2
        mx, my = to_xy(coords[mid, 1], coords[mid, 0])
        # keep labels a bit inside the edge so they are not clipped
        if xlim[0] + 40 < mx < xlim[1] - 40 and ylim[0] + 40 < my < ylim[1] - 40:
            ang = np.degrees(np.arctan2(coords[mid+1, 1] - coords[mid-1, 1],
                                        coords[mid+1, 0] - coords[mid-1, 0]))
            if ang > 90:
                ang -= 180
            if ang < -90:
                ang += 180
            ax.text(mx, my, name, fontsize=7, rotation=ang, ha='center', va='center', zorder=4)
            labeled.add(name)

# Draw the trajectory, colored by label.
labels = {0: ('walk', 'tab:blue'), 1: ('bike', 'tab:green'),
          2: ('car', 'tab:orange'), 3: ('jam', 'tab:red')}
for l, (name, color) in labels.items():
    sub = df[df.label == l]
    x, y = to_xy(sub.true_lat, sub.true_lon)
    ax.scatter(x, y, s=0.7, alpha=0.4, color=color, label=name, zorder=2)

# The gNB and the "ESS" reference point (see below).
gnb_lat, gnb_lon = 55.71622, 13.22581
ess_lat, ess_lon = 55.72487, 13.23098
gx, gy = to_xy(gnb_lat, gnb_lon)
ex, ey = to_xy(ess_lat, ess_lon)
ax.plot(gx, gy, 'k^', ms=13, label='gNB (radionode 010)', zorder=5)
ax.plot(ex, ey, 'ks', ms=11, label='ESS reference point', zorder=5)

ax.set_aspect(1)
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim(xlim)
ax.set_ylim(ylim)

ax.set_title('Trajectory of the trace on the OpenStreetMap road network, Lund')
plt.tight_layout()
plt.show()

Two fixed points are referenced in the data:

* the **gNB** (`radionode 010`), on one of the Ericsson campus buildings. We can
  locate it from the data itself: the row with the smallest `radionode010` is
  the point where the UE came closest to the base station, so the gNB is within
  a few tens of meters of that position.
* the **ESS reference point**, used for the `gps_dist_ess` column (the GPS
  distance from the UE to this point). It is a communication tower at the north
  end of the campus area, and can be located the same way, using the smallest
  `gps_dist_ess`.

In [ ]:
closest_to_gnb = df.loc[df.radionode010.idxmin()]
print(f'Closest approach to the gNB: {closest_to_gnb.radionode010:.1f} m away, '
      f'at lat {closest_to_gnb.true_lat:.5f}, lon {closest_to_gnb.true_lon:.5f}')
closest_to_ess = df.loc[df.gps_dist_ess.idxmin()]
print(f'Closest approach to the ESS point: {closest_to_ess.gps_dist_ess:.1f} m away, '
      f'at lat {closest_to_ess.true_lat:.5f}, lon {closest_to_ess.true_lon:.5f}')

The data also gives the raw GPS position (`lat`/`lon`) and the ground truth
position (`true_lat`/`true_lon`). As the dataset documentation points out, the
GPS fixes are at times by no means accurate. Let's look at how far the raw GPS
fixes are from the ground truth.

In [ ]:
fig, ax = plt.subplots()
d_gps = np.hypot(
    (df.lat - df.true_lat) * 110574,
    (df.lon - df.true_lon) * 111320 * np.cos(np.radians(df.lat)))
ax.hist(d_gps, bins=80, range=(0, 60), color='tab:blue', alpha=0.8)
ax.set_xlabel('distance between raw GPS and ground truth (m)')
ax.set_ylabel('number of records')
ax.set_title(f'Median error {np.median(d_gps):.1f} m, 90th percentile {np.percentile(d_gps, 90):.1f} m')
plt.tight_layout()
plt.show()

## The journey: speed and labels in time

Now let's follow the journey in the time domain: the speed of the UE and the
label of the movement scenario.

In [ ]:
df['time'] = pd.to_datetime(df.datetime)
df = df.sort_values('time')

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(12, 6),
                               gridspec_kw={'height_ratios': [3, 1]})
ax1.plot(df.time, df.speed_smooth, color='0.3', lw=0.5)
ax1.set_ylabel('speed (km/h)')
ax1.set_ylim(0, 80)
ax1.set_title('Speed and movement label over time')
ax2.step(df.time, df.label, where='post', color='0.3', lw=1.0)
ax2.set_yticks([0, 1, 2, 3], [l[0] for l in labels.values()])
ax2.set_ylabel('label')
plt.tight_layout()
plt.show()

The trace starts in the middle of the campus and drives north (`car`), towards
the ESS reference point. There is a `jam` at the far end of the road, after
which the car drives back south, with a short `bike` period (a car at 10-15
km/h on a bike road) in the middle of the campus, and finally a long `walk`
around the southern part of the campus, close to the gNB.

Let's check that the labels match the speed:

In [ ]:
print('Median speed per label (km/h):')
df.groupby('label').speed_smooth.median().rename('median speed (km/h)')

## The radio side: timing advance

The timing advance is the central feature of this dataset. The gNB tells the
UE how much to advance its uplink transmission; the value is proportional to
the round-trip propagation time, and hence to the distance between the UE and
the gNB. In the data:

* `radionode010` is the distance (m) from the UE to the gNB,
* `distanceTa` is a distance (m) estimated from the TA measurements. It is
  quantized, in steps of about 40 m, which is the resolution of the TA, and it
  deviates from the true distance by tens of meters,
* the other `ta*` columns are the raw and filtered TA measurements that a model
  would be trained on.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(12, 6),
                               gridspec_kw={'height_ratios': [3, 1]})
ax1.plot(df.time, df.radionode010, color='0.3', lw=0.5)
ax1.set_ylabel('distance to gNB (m)')
ax1.set_title('Distance to the gNB over time')
ax2.step(df.time, df.label, where='post', color='0.3', lw=1.0)
ax2.set_yticks([0, 1, 2, 3], [l[0] for l in labels.values()])
ax2.set_ylabel('label')
plt.tight_layout()
plt.show()

The distance to the gNB follows the trajectory: it grows while the UE drives
away from the campus and shrinks while it drives back.

Compare it with the distance estimated from the TA:

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df.radionode010, df.distanceTa, s=0.5, alpha=0.3, color='tab:blue')
lim = [0, df.radionode010.max()]
ax.plot(lim, lim, 'k--', lw=1)
ax.set_xlabel('distance to gNB from GPS (m)')
ax.set_ylabel('distance to gNB from TA (m)')
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_title('Distance from the TA vs the true distance')
plt.tight_layout()
plt.show()

The TA-based distance follows the true distance, but it is quantized in steps
of about 40 m and has an error of the order of 50-100 m. Estimating the
distance (or the movement) from the TA is therefore a non-trivial problem, and
that is what the rest of the dataset is about.

Let's look at the raw TA measurements that a model would use. Each row comes
from one of a few radio record types, given by the `trace` column:

In [ ]:
df.trace.value_counts()

In [ ]:
# Zoom in on a 5 minute window, 20 minutes after the start of the trace.
t0 = df.time[0] + np.timedelta64(20, 'm')
t1 = t0 + np.timedelta64(5, 'm')
sub = df[(df.time >= t0) & (df.time <= t1)]

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(12, 6))
s1 = sub[sub.trace == 'ULL1COMMONNR.1202']
ax1.plot(s1.time, s1.taMeasHighRes, '.', ms=2, alpha=0.5, color='tab:blue')
ax1.set_ylabel('taMeasHighRes')
s2 = sub[sub.trace == 'UPCUEDLNR.854']
ax2.plot(s2.time, s2.filteredTA, '.', ms=2, alpha=0.5, color='tab:orange')
ax2.set_ylabel('filteredTA')
ax2.set_title('Raw TA measurements over a 5 minute window')
plt.tight_layout()
plt.show()

The raw measurements are noisy and only present in parts of the time (different
radio record types report at different rates). A model has to learn to
relate these measurements, together with their temporal structure, to the
movement of the UE.

## A first classifier

Let's build simple classifiers for the movement label. First a baseline that
uses the GPS-derived speed and the road information, and then a second one that
uses only the radio measurements, to see how far radio data alone gets us.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
import seaborn as sns

# Rate of change of the distance to the gNB and of the TA-based distance,
# over a 10 s window.
dt10 = df.time.diff(10).dt.total_seconds().fillna(0)
df['d_gnb_dt'] = df.radionode010.diff(10) / dt10
df['d_ta_dt'] = df.distanceTa.diff(10) / dt10
df = df.replace([np.inf, -np.inf], np.nan)

gps_features = ['speed_smooth', 'speed_limit', 'carway', 'bikeway', 'walkway',
                'radionode010', 'd_gnb_dt']
radio_features = ['distanceTa', 'd_ta_dt', 'taMeasHighRes', 'filteredTA',
                  'scaledTA', 'timingOffsetFloat']

def evaluate(features, name):
    X = StandardScaler().fit_transform(df[features].fillna(0))
    y = df.label
    model = LogisticRegression(max_iter=2000)
    model.fit(X, y)
    pred = model.predict(X)
    print(f'{name}: accuracy on training data {accuracy_score(y, pred):.3f}\n')
    return pred

pred_gps = evaluate(gps_features, 'GPS-based (speed + road info)')
pred_radio = evaluate(radio_features, 'Radio-based (TA features only)')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(confusion_matrix(df.label, pred_gps), annot=True, fmt='d',
            cmap='Blues', xticklabels=[l[0] for l in labels.values()],
            yticklabels=[l[0] for l in labels.values()])
ax.set_xlabel('predicted')
ax.set_ylabel('true')
ax.set_title('Baseline classifier (GPS-based features)')
plt.tight_layout()
plt.show()

The baseline, which uses the GPS-derived speed and the road information, is
very good, because the speed alone almost separates the scenarios. The
radio-based classifier is much weaker: the TA measurements only give a
rough, quantized distance to the gNB, which does not by itself determine how
the UE is moving.

This illustrates the problem the dataset is meant for: to learn, from the radio
measurements alone, how the UE is moving in the landscape. A full pipeline
would combine both files of the dataset, use more elaborate features from the
TA (for example its temporal structure), and validate the model properly on
held-out data.

## Summary

* The `R-Traffic` dataset combines GPS traces with 5G radio measurements
  (timing advance) and labels of the movement scenario, collected around the
  Ericsson buildings in Lund.
* The trajectory can be visualized on the OpenStreetMap road network of the
  area; the gNB can be located from the data itself, as the point where the
  distance to it is smallest.
* The raw GPS fixes deviate from the ground truth, sometimes by 10 m or
  more, so the GPS data itself is noisy.
* The TA gives a quantized distance to the gNB with an error of tens of
  meters; relating the raw TA measurements to the movement of the UE is a
  machine learning problem.
* A GPS-based baseline classifier is nearly perfect, while a classifier based
  on the radio measurements alone is far from it, which is the challenge the
  dataset is meant to address.